<p style="text-align:right" >
    <a href="https://www.iesb.br/home/" target="_blank">
    <img src="https://www.iesb.br/content/themes/iesb-chleba-themosis/assets/imagens/logo.png" width="100"/>
    </a>
</p>

# 🧠 Inteligência Artificial - IESB

**Tema da aula:** Laboratório 2 - Busca em Espaço de Estados
**Docente:** Prof. Rodrigo Gonçalves Pinto
**Ambiente:** Google Colab

---

## 🎯 Objetivos de Aprendizagem
Neste laboratório, você vai:
1. **Representar um espaço de estados:** Transformar um mapa rodoviário em uma estrutura de dados Python.
2. **Construir a máquina de busca:** Implementar o nó, a fronteira, o conjunto de explorados e a reconstrução do caminho.
3. **Descobrir que a fronteira é o algoritmo:** Trocar fila por fila de prioridade e ver o comportamento mudar completamente.
4. **Implementar e comparar quatro buscas:** Largura, custo uniforme, gulosa e A\*, medindo custo e esforço de cada uma.

---

## ⚠️ Antes de começar
> **Crie sua cópia:** Vá em `Arquivo → Salvar uma cópia no Drive`. Trabalhe sempre na sua cópia, pois as edições no arquivo original não são salvas!

---

## 💡 Como aproveitar este notebook
- O conteúdo intercala **explicações teóricas** com **código comentado**.
- Execute as células na ordem (usando `Shift + Enter`).
- **Não pule o código pronto:** metade do aprendizado está em entender o esqueleto antes de preencher a lacuna.
- Se um TODO ficar em branco, as células seguintes avisam e seguem em frente. Nada quebra.

## 🛠️ Pré-requisitos
- Laboratório 1 concluído (tuplas, listas, dicionários, funções e pandas).
- Aula teórica sobre formulação de problemas e busca em espaço de estados.

> *"Todo algoritmo de busca é o mesmo laço. Muda só quem sai da fila primeiro."*

---
# Etapa 1 — Do mapa ao código

## O problema desta aula

Um agente está em **Arad**, no oeste da Romênia, e precisa chegar a **Bucareste**. Ele tem o mapa: sabe quais cidades existem, quais estradas ligam quais cidades e quantos quilômetros tem cada trecho. Ninguém disse a ele qual caminho seguir.

Na aula teórica vimos que um problema de busca tem **cinco componentes**. O que talvez tenha passado despercebido é que uma única estrutura de dados Python carrega **quatro deles**.

## O mapeamento de conceitos

Assim como no Laboratório 1 cada estrutura Python representava um conceito de agente, aqui cada uma representa um componente do problema de busca:

| Componente do problema | Como aparece no código |
|---|---|
| **Estado inicial** | Uma string: `"Arad"` |
| **Ações** | `MAPA[cidade].keys()` — as cidades para onde dá para ir |
| **Modelo de transição** | A própria chave vizinha: ir para `"Sibiu"` resulta no estado `"Sibiu"` |
| **Teste de objetivo** | `cidade == "Bucareste"` |
| **Custo de passo** | `MAPA[cidade][vizinho]` — o número em quilômetros |

## A estrutura escolhida: dicionário de dicionários

A chave externa é a cidade. O valor é **outro dicionário**, em que a chave é uma cidade vizinha e o valor é a distância da estrada entre as duas.

```python
MAPA["Arad"]              # {'Zerind': 75, 'Sibiu': 140, 'Timisoara': 118}
MAPA["Arad"]["Sibiu"]     # 140
```

> **Atenção à ordem.** A partir do Python 3.7, dicionários preservam a ordem de inserção. Isso importa muito aqui: a ordem em que os vizinhos aparecem decide qual caminho a busca em profundidade vai seguir. Manter a ordem do livro é o que faz seus resultados baterem com os valores de referência.

In [ ]:
# ---------------------------------------------------------------------
# O mapa rodoviário da Romênia, como dicionário de dicionários.
#
# Fonte dos dados: Russell & Norvig, Inteligência Artificial, 3ª ed.,
# Figura 3.2. O grafo é NÃO DIRIGIDO: se existe estrada de A para B,
# existe de B para A, com a mesma distância.
# ---------------------------------------------------------------------

MAPA = {
    "Arad":           {"Zerind": 75, "Sibiu": 140, "Timisoara": 118},
    "Zerind":         {"Arad": 75, "Oradea": 71},
    "Oradea":         {"Zerind": 71, "Sibiu": 151},
    "Sibiu":          {"Arad": 140, "Oradea": 151, "Fagaras": 99,
                       "Rimnicu Vilcea": 80},
    "Timisoara":      {"Arad": 118, "Lugoj": 111},
    "Lugoj":          {"Timisoara": 111, "Mehadia": 70},
    "Mehadia":        {"Lugoj": 70, "Drobeta": 75},
    "Drobeta":        {"Mehadia": 75, "Craiova": 120},
    "Craiova":        {"Drobeta": 120, "Rimnicu Vilcea": 146, "Pitesti": 138},
    "Rimnicu Vilcea": {"Sibiu": 80, "Craiova": 146, "Pitesti": 97},
    "Fagaras":        {"Sibiu": 99, "Bucareste": 211},
    "Pitesti":        {"Rimnicu Vilcea": 97, "Craiova": 138, "Bucareste": 101},
    "Bucareste":      {"Fagaras": 211, "Pitesti": 101, "Giurgiu": 90,
                       "Urziceni": 85},
    "Giurgiu":        {"Bucareste": 90},
    "Urziceni":       {"Bucareste": 85, "Hirsova": 98, "Vaslui": 142},
    "Hirsova":        {"Urziceni": 98, "Eforie": 86},
    "Eforie":         {"Hirsova": 86},
    "Vaslui":         {"Urziceni": 142, "Iasi": 92},
    "Iasi":           {"Vaslui": 92, "Neamt": 87},
    "Neamt":          {"Iasi": 87},
}

# len() de um dicionário devolve quantas CHAVES ele tem: aqui, as cidades.
print("Cidades no mapa:", len(MAPA))

# Cada estrada aparece duas vezes (uma em cada ponta), por isso dividimos por 2.
# A expressão dentro do sum() é uma "expressão geradora": ela soma o tamanho
# de cada dicionário interno sem precisar montar uma lista antes.
print("Estradas no mapa:", sum(len(v) for v in MAPA.values()) // 2)

### Auditando o mapa antes de usá-lo

No Laboratório 1 vimos que reprodutibilidade vale nota. Aqui vale um princípio parecido: **algoritmo nenhum valida os dados de entrada**. Uma busca perfeita sobre um mapa com erro devolve um caminho errado, sem reclamar de nada.

Antes de qualquer busca, vamos verificar duas propriedades que o mapa **precisa** ter:

1. **Simetria:** se `MAPA["Arad"]["Sibiu"]` vale 140, então `MAPA["Sibiu"]["Arad"]` também tem que valer 140.
2. **Fechamento:** toda cidade citada como vizinha precisa existir como chave no dicionário.

Um erro de digitação em qualquer uma das 46 entradas quebra pelo menos uma dessas regras.

In [ ]:
# ---------------------------------------------------------------------
# AUDITORIA DO MAPA — duas verificações antes de confiar nos dados.
#
# Variável:
#   problemas -> lista que acumula as inconsistências encontradas
# ---------------------------------------------------------------------

problemas = []                      # a lista começa vazia: nenhum erro ainda

# .items() percorre o dicionário devolvendo o par (chave, valor) de cada vez.
for cidade, vizinhos in MAPA.items():
    for vizinho, distancia in vizinhos.items():

        # Verificação 1: a cidade vizinha existe como chave no mapa?
        if vizinho not in MAPA:
            problemas.append(f"{vizinho} aparece como vizinha mas não existe")

        # Verificação 2: a distância é a mesma nos dois sentidos?
        # .get() consulta com segurança: devolve None se a chave não existir,
        # em vez de interromper o programa com KeyError.
        elif MAPA[vizinho].get(cidade) != distancia:
            problemas.append(f"{cidade}-{vizinho}: assimetria de distância")

if problemas:
    print("Problemas encontrados:")
    for p in problemas:
        print("  -", p)
else:
    print("Mapa íntegro: 20 cidades, 23 estradas, todas simétricas.")

### As três consultas que a busca faz ao mapa

Um algoritmo de busca faz três perguntas ao mapa, e só três. Vale rodar a célula abaixo devagar e reconhecer cada uma — é literalmente o que o `for` do seu algoritmo vai fazer daqui a pouco.

In [ ]:
# ---------------------------------------------------------------------
# As três consultas que todo algoritmo de busca faz ao mapa.
# ---------------------------------------------------------------------

cidade_atual = "Sibiu"

# PERGUNTA 1 — "para onde eu posso ir a partir daqui?"
# list() converte as chaves em lista, só para imprimir de forma legível.
print("Vizinhos de", cidade_atual, "->", list(MAPA[cidade_atual].keys()))

# PERGUNTA 2 — "quanto custa cada uma dessas estradas?"
print("Custos          ->", MAPA[cidade_atual])

# PERGUNTA 3 — "quanto custa ir daqui para uma cidade específica?"
print("Sibiu -> Fagaras:", MAPA["Sibiu"]["Fagaras"], "km")

print()   # print() sem argumentos apenas pula uma linha

# O padrão que vai aparecer DENTRO de todos os algoritmos:
# percorrer os vizinhos e seus custos ao mesmo tempo, com .items().
# A f-string usa :16s para alinhar o nome em 16 caracteres, deixando
# a saída em colunas; :3d faz o mesmo com um número inteiro.
for vizinho, custo in MAPA[cidade_atual].items():
    print(f"  de {cidade_atual} para {vizinho:16s} custa {custo:3d} km")

---
## 🛠️ TODO 1 — O custo de um caminho

Um **caminho** é uma lista de cidades, como `["Arad", "Sibiu", "Fagaras", "Bucareste"]`. O **custo do caminho** é a soma dos custos de cada trecho.

Escreva a função `custo_do_caminho(caminho)`, que percorre a lista somando as distâncias.

**Dica:** você precisa dos pares consecutivos da lista. Um jeito simples é usar `range(len(caminho) - 1)` e acessar `caminho[i]` e `caminho[i + 1]`.

In [ ]:
# =====================================================================
# TODO 1 — Complete a função abaixo.
#
# custo_do_caminho(caminho) recebe uma lista de cidades e devolve a
# soma dos custos de cada trecho percorrido.
#
# Exemplo: ["Arad", "Sibiu"] deve devolver 140.
# =====================================================================

def custo_do_caminho(caminho):
    """Soma os custos de passo ao longo de uma lista de cidades.

    Parâmetro:
        caminho (list): lista de cidades em ordem, ex. ["Arad", "Sibiu"].

    Devolve:
        int: o custo total do caminho, em quilômetros.
    """
    # Apague a linha abaixo e escreva sua solução.
    return None


# ------------------- teste automático, não precisa alterar -------------------
rota_fagaras = ["Arad", "Sibiu", "Fagaras", "Bucareste"]
rota_pitesti = ["Arad", "Sibiu", "Rimnicu Vilcea", "Pitesti", "Bucareste"]

resultado = custo_do_caminho(rota_fagaras)

if resultado is None:
    print("TODO 1 ainda não foi resolvido.")
elif resultado == 450 and custo_do_caminho(rota_pitesti) == 418:
    # Conferimos os dois casos, para garantir que a função realmente soma
    # e não está devolvendo um número fixo.
    print("TODO 1 correto!")
    print("  Rota por Fagaras:", resultado, "km")
    print("  Rota por Pitesti:", custo_do_caminho(rota_pitesti), "km")
    print()
    print("Duas rotas válidas para o mesmo problema, com custos diferentes.")
    print("A diferença é de", resultado - custo_do_caminho(rota_pitesti), "km.")
else:
    print("Resultado inesperado:", resultado,
          "- esperado 450 para a rota por Fagaras.")

---
# Etapa 2 — Vendo o mapa

Números em um dicionário não constroem intuição geográfica. A célula abaixo desenha o mapa com `matplotlib`, usando um dicionário de coordenadas aproximadas.

Você **não precisa entender esta função linha a linha** — ela é uma ferramenta de visualização, não conteúdo de prova. Mas vale ler os comentários: no fim do notebook você vai reusá-la para pintar o caminho encontrado por cada algoritmo.

In [ ]:
# ---------------------------------------------------------------------
# Ferramenta de visualização: desenha o mapa e, opcionalmente, destaca
# um caminho e numera os nós expandidos por um algoritmo.
#
# Parâmetros:
#   caminho    (list) -> cidades a pintar como solução
#   expandidos (list) -> cidades expandidas, na ordem, para numerar
#   titulo     (str)  -> título do gráfico
# ---------------------------------------------------------------------

import matplotlib.pyplot as plt

# Coordenadas aproximadas de cada cidade, só para o desenho lembrar a
# geografia real. Elas não têm nenhum papel no algoritmo de busca.
COORD = {
    "Arad": (91, 492), "Bucareste": (400, 327), "Craiova": (253, 288),
    "Drobeta": (165, 299), "Eforie": (562, 293), "Fagaras": (305, 449),
    "Giurgiu": (375, 270), "Hirsova": (534, 350), "Iasi": (473, 506),
    "Lugoj": (165, 379), "Mehadia": (168, 339), "Neamt": (406, 537),
    "Oradea": (131, 571), "Pitesti": (320, 368), "Rimnicu Vilcea": (233, 410),
    "Sibiu": (207, 457), "Timisoara": (94, 410), "Urziceni": (456, 350),
    "Vaslui": (509, 444), "Zerind": (108, 531),
}

VERMELHO_IESB = "#C8102E"


def desenhar_mapa(caminho=None, expandidos=None, titulo="Mapa da Romênia"):
    """Desenha o mapa; se receber um caminho, pinta-o em vermelho."""
    caminho = caminho or []          # "or []" evita erro quando vem None
    expandidos = expandidos or []

    fig, ax = plt.subplots(figsize=(13, 8))

    # 1) todas as estradas em cinza claro, ao fundo
    ja_desenhadas = set()
    for cidade, vizinhos in MAPA.items():
        for vizinho in vizinhos:
            # o par ordenado evita desenhar a mesma estrada duas vezes
            par = tuple(sorted([cidade, vizinho]))
            if par in ja_desenhadas:
                continue
            ja_desenhadas.add(par)
            x1, y1 = COORD[cidade]
            x2, y2 = COORD[vizinho]
            ax.plot([x1, x2], [y1, y2], color="#DDE2E7", lw=1.6, zorder=1)

    # 2) por cima, as estradas que fazem parte do caminho-solução.
    # zip(caminho, caminho[1:]) forma os pares consecutivos da lista:
    # ["A","B","C"] vira ("A","B") e ("B","C").
    for a, b in zip(caminho, caminho[1:]):
        x1, y1 = COORD[a]
        x2, y2 = COORD[b]
        ax.plot([x1, x2], [y1, y2], color=VERMELHO_IESB, lw=5, alpha=0.45,
                zorder=2)
        ax.text((x1 + x2) / 2, (y1 + y2) / 2, str(MAPA[a][b]), fontsize=10,
                color=VERMELHO_IESB, ha="center", va="center", zorder=5,
                bbox=dict(boxstyle="round,pad=0.2", fc="white",
                          ec=VERMELHO_IESB))

    # 3) as cidades
    for cidade, (x, y) in COORD.items():
        if cidade in expandidos:
            ordem = expandidos.index(cidade) + 1     # posição na expansão
            ax.scatter(x, y, s=330, color=VERMELHO_IESB, zorder=3)
            ax.text(x, y, str(ordem), fontsize=10, color="white", ha="center",
                    va="center", fontweight="bold", zorder=4)
        else:
            ax.scatter(x, y, s=90, color="white", edgecolors="#1F2933",
                       linewidths=1.4, zorder=3)
        destaque = cidade in caminho or cidade in expandidos
        ax.text(x, y + 18, cidade, fontsize=10 if destaque else 9,
                color="#1F2933" if destaque else "#8A94A0", ha="center",
                fontweight="bold" if destaque else "normal", zorder=6)

    ax.set_title(titulo, fontsize=15, fontweight="bold", loc="left")
    ax.axis("off")
    plt.tight_layout()
    plt.show()


desenhar_mapa(titulo="O espaço de estados: 20 cidades, 23 estradas")

---
# Etapa 3 — O nó, a fronteira e os explorados

## Estado não é nó

Esta distinção parece burocrática e não é: ela é a causa do bug mais comum em código de busca.

- **Estado** é uma configuração do mundo. Aqui, uma cidade: `"Pitesti"`.
- **Nó** é uma estrutura de dados **da busca**. Ele guarda o estado, mas também guarda *como se chegou até ele*.

Por que a diferença importa? Porque existem vários caminhos até Pitesti. Chegar por Rimnicu Vilcea custa 317 km; chegar pelo sul, via Craiova, custa 632 km. São **dois nós diferentes com o mesmo estado** — e é escolhendo entre eles que a busca encontra o caminho ótimo.

## O mapeamento de conceitos

| Conceito da aula teórica | Implementação em Python |
|---|---|
| Estado | Uma string: `"Pitesti"` |
| Nó | Um objeto da classe `No` |
| Custo de caminho g(n) | O atributo `no.g` |
| Fronteira | Uma fila (`deque`) ou uma fila de prioridade (`heapq`) |
| Conjunto de explorados | Um `set` de strings |
| Solução | Lista de cidades devolvida por `reconstruir_caminho` |

> **O campo `pai` é o coração da coisa.** Nenhum algoritmo guarda o caminho inteiro dentro de cada nó — isso repetiria o mesmo trecho inicial em milhares de nós. Guarda-se um ponteiro para o pai e, quando o objetivo aparece, sobe-se de pai em pai até a raiz.

In [ ]:
# ---------------------------------------------------------------------
# A estrutura NÓ.
#
# Usamos uma classe com __slots__ para deixar explícitos os quatro
# campos — e, de quebra, gastar menos memória, o que importa quando a
# busca gera milhares de nós.
# ---------------------------------------------------------------------

class No:
    """Um nó da árvore de busca: estado + memória de como chegamos até ele."""

    __slots__ = ("estado", "pai", "acao", "g")

    def __init__(self, estado, pai=None, acao=None, g=0):
        self.estado = estado    # a cidade
        self.pai = pai          # o nó anterior (None se for a raiz)
        self.acao = acao        # a ação aplicada no pai
        self.g = g              # custo acumulado desde o início

    def __repr__(self):
        # __repr__ define o que aparece quando imprimimos o objeto.
        # Sem ele, o Python mostraria algo como <__main__.No object at 0x7f...>.
        return f"No({self.estado}, g={self.g})"

    def __lt__(self, outro):
        # __lt__ define o operador "<". O heapq (fila de prioridade) precisa
        # dele para desempatar quando duas prioridades forem iguais.
        return self.g < outro.g


# Montando uma cadeia de nós NA MÃO, para ver a estrutura funcionando.
raiz    = No("Arad")
n_sibiu = No("Sibiu",          pai=raiz,    acao="Sibiu",          g=140)
n_rv    = No("Rimnicu Vilcea", pai=n_sibiu, acao="Rimnicu Vilcea", g=220)
n_pit   = No("Pitesti",        pai=n_rv,    acao="Pitesti",        g=317)

print("O nó final:", n_pit)
print("O pai dele:", n_pit.pai)
print("O avô dele:", n_pit.pai.pai)
print()
print("Custo acumulado até Pitesti:", n_pit.g, "km")
print("Conferindo na mão: 140 + 80 + 97 =", 140 + 80 + 97)

### Reconstruindo o caminho de trás para frente

Quando o algoritmo encontra o objetivo, ele tem em mãos **um único nó**. O caminho não está guardado em lugar nenhum — está implícito na cadeia de pais.

Reconstruí-lo é um laço `while` de quatro linhas: comece pelo nó final, anote o estado, suba para o pai, repita até o pai ser `None`. No fim, a lista está de trás para frente, e basta invertê-la.

In [ ]:
# ---------------------------------------------------------------------
# Reconstrução do caminho: do nó-objetivo até a raiz, seguindo os pais.
# ---------------------------------------------------------------------

def reconstruir_caminho(no):
    """Devolve a lista de estados da raiz até o nó recebido.

    Parâmetro:
        no (No): o nó final, normalmente o que satisfez o teste de objetivo.

    Devolve:
        list: as cidades em ordem, do início até o nó.
    """
    caminho = []                    # lista vazia para acumular os estados

    while no is not None:           # "is not None" para quando chegar na raiz
        caminho.append(no.estado)   # anota o estado do nó atual
        no = no.pai                 # sobe um nível na cadeia

    # Neste ponto a lista está de TRÁS PARA FRENTE: [Pitesti, ..., Arad].
    # A fatia [::-1] percorre a lista de ré e devolve uma cópia invertida.
    return caminho[::-1]


caminho_reconstruido = reconstruir_caminho(n_pit)

print("Caminho reconstruído:", caminho_reconstruido)
print("Custo guardado no nó:", n_pit.g, "km")

# Reusando o TODO 1: o custo somado do caminho tem que bater com o g do nó.
if custo_do_caminho(caminho_reconstruido) is not None:
    print("Custo recalculado   :", custo_do_caminho(caminho_reconstruido), "km")
else:
    print("(complete o TODO 1 para conferir o custo por outro caminho)")

### A fronteira decide o algoritmo

Esta é a ideia central do laboratório inteiro.

Os quatro algoritmos que você vai rodar seguem **exatamente o mesmo ciclo**:

```
retire um nó da fronteira
é o objetivo?  →  pare
marque o estado como explorado
gere os vizinhos ainda não explorados
insira-os na fronteira
```

O que muda de um algoritmo para o outro é uma coisa só: **qual nó a fronteira devolve quando pedimos o próximo**.

| Estrutura da fronteira | Devolve o nó... | Algoritmo |
|---|---|---|
| Fila (o primeiro que entrou é o primeiro que sai) | mais antigo | Busca em largura |
| Pilha (o último que entrou é o primeiro que sai) | mais recente | Busca em profundidade |
| Fila de prioridade ordenada por `g` | mais barato até aqui | Busca de custo uniforme |
| Fila de prioridade ordenada por `h` | que parece mais perto | Busca gulosa |
| Fila de prioridade ordenada por `g + h` | de melhor estimativa total | **A\*** |

> **Convenção da disciplina:** o teste de objetivo acontece quando o nó é **removido** da fronteira, nunca quando ele é gerado. Isso vale para todos os algoritmos aqui. Sem essa convenção, o custo uniforme devolveria 450 km em vez de 418 — e a contagem de nós expandidos deixaria de ser comparável entre algoritmos.

---
# Etapa 4 — Busca em largura

## A fila

A busca em largura usa uma **fila**: o primeiro nó que entra é o primeiro que sai. O efeito é explorar o mapa em camadas — todas as cidades a uma estrada de distância, depois todas a duas, e assim por diante.

Em Python usamos `collections.deque`, que é uma fila otimizada:

```python
from collections import deque
fila = deque()
fila.append(x)      # insere no FIM
fila.popleft()      # remove do INÍCIO   <- é isto que faz a regra ser de fila
```

Se você usasse `fila.pop()` (que remove do fim), teria escrito uma **pilha** — e o algoritmo viraria busca em profundidade. Uma palavra de diferença.

---
## 🛠️ TODO 2 — Complete a busca em largura

O esqueleto abaixo está completo, **exceto por uma linha**: a que insere o nó vizinho na fronteira.

**Resultado esperado:** `Arad → Sibiu → Fagaras → Bucareste`, custo **450 km**, **8** nós expandidos.

In [ ]:
# =====================================================================
# TODO 2 — Complete a busca em largura.
#
# Falta UMA linha: inserir o nó vizinho na fronteira.
# Dica: a fronteira é um deque. Para inserir no fim, use .append()
# =====================================================================

from collections import deque


def busca_largura(inicio, objetivo):
    """Busca em largura em grafo.

    Parâmetros:
        inicio (str): cidade de partida.
        objetivo (str): cidade de destino.

    Devolve:
        tuple: (caminho, custo, nós expandidos, ordem de expansão).
    """
    fronteira = deque([No(inicio)])   # a fronteira começa com o nó inicial
    explorados = set()                # conjunto: consulta imediata
    na_fronteira = {inicio}           # espelho da fronteira, para consulta rápida
    ordem = []                        # registra a ordem das expansões

    while fronteira:                  # enquanto houver nós para examinar
        no = fronteira.popleft()      # retira o mais antigo: regra da fila
        na_fronteira.discard(no.estado)

        # Teste de objetivo NA REMOÇÃO, conforme a convenção da disciplina.
        if no.estado == objetivo:
            return reconstruir_caminho(no), no.g, len(ordem), ordem

        explorados.add(no.estado)     # marca como já expandido
        ordem.append(no.estado)

        for vizinho, custo in MAPA[no.estado].items():
            if vizinho in explorados or vizinho in na_fronteira:
                continue              # já visto: ignora e vai para o próximo

            filho = No(vizinho, pai=no, acao=vizinho, g=no.g + custo)

            # -------------------------------------------------------------
            # TODO 2: insira o nó "filho" no FIM da fronteira.
            # Apague a linha "pass" e escreva sua solução no lugar.
            # -------------------------------------------------------------
            pass

            na_fronteira.add(vizinho)

    return None, None, len(ordem), ordem


# ------------------- teste automático, não precisa alterar -------------------
cam_bfs, custo_bfs, exp_bfs, ordem_bfs = busca_largura("Arad", "Bucareste")

if cam_bfs is None:
    print("TODO 2 ainda não foi resolvido: a busca não encontrou caminho.")
elif custo_bfs == 450 and exp_bfs == 8:
    print("TODO 2 correto!")
    print("  Caminho:", " -> ".join(cam_bfs))
    print("  Custo:", custo_bfs, "km  |  Nós expandidos:", exp_bfs)
else:
    print("Resultado inesperado: custo", custo_bfs, "e", exp_bfs,
          "expandidos - esperado: custo 450 e 8 expandidos.")

### Lendo o resultado

A busca em largura devolveu um caminho com **três estradas** — e é isso que ela otimiza: o **número de ações**, não a soma dos custos.

Rode a célula abaixo para ver o caminho desenhado, com os nós numerados na ordem em que foram expandidos. Repare que ela expandiu Zerind e Timisoara, que ficam a oeste, na direção **oposta** a Bucareste. A busca em largura não tem como saber disso: para ela, o mapa não tem geografia.

**Guarde o número 450.** Na próxima etapa ele vai virar 418.

In [ ]:
# ---------------------------------------------------------------------
# Visualizando o que a busca em largura fez.
# ---------------------------------------------------------------------

if cam_bfs:                                 # só desenha se o TODO 2 funcionou
    desenhar_mapa(
        caminho=cam_bfs,
        expandidos=ordem_bfs,
        titulo=f"Busca em largura - {custo_bfs} km, {exp_bfs} nós expandidos",
    )
    print("Ordem de expansão:", " · ".join(ordem_bfs))
else:
    print("Complete o TODO 2 para ver este gráfico.")

---
# Etapa 5 — Busca de custo uniforme

## A fila de prioridade

A busca de custo uniforme troca a fila comum por uma **fila de prioridade**: em vez de devolver o nó mais antigo, ela devolve o **mais barato até aqui** — o de menor `g`.

Em Python usamos o módulo `heapq`, que implementa uma fila de prioridade sobre uma lista comum:

```python
import heapq
heap = []
heapq.heappush(heap, (prioridade, item))   # insere
prioridade, item = heapq.heappop(heap)     # remove SEMPRE o de menor prioridade
```

O truque é empurrar **tuplas** `(prioridade, contador, nó)`. O contador serve de desempate: se duas prioridades forem iguais, o Python tentaria comparar os nós entre si, e sem critério de desempate o resultado dependeria da ordem de inserção. Com o contador, quem entrou primeiro sai primeiro — e o resultado é reprodutível, exatamente como a semente `random.seed(42)` do Laboratório 1.

## Por que aqui um estado pode entrar duas vezes na fronteira

Diferente da busca em largura, esta **permite** que o mesmo estado entre mais de uma vez na fronteira, com custos diferentes. Isso não é um bug: se um caminho mais barato para uma cidade for descoberto depois, ele precisa ter chance de ser expandido antes. Quando um estado repetido é finalmente retirado, o `if no.estado in explorados: continue` o descarta.

---
## 🛠️ TODO 3 — Complete a busca de custo uniforme

Falta **uma linha**: empurrar o filho no heap com a prioridade correta.

**Resultado esperado:** `Arad → Sibiu → Rimnicu Vilcea → Pitesti → Bucareste`, custo **418 km**, **12** nós expandidos.

In [ ]:
# =====================================================================
# TODO 3 — Complete a busca de custo uniforme.
#
# Falta UMA linha: empurrar o filho no heap com a prioridade correta.
# Dica: nesta busca a prioridade é o custo acumulado, ou seja, filho.g
# Formato: heapq.heappush(fronteira, (prioridade, next(contador), filho))
# =====================================================================

import heapq
import itertools


def busca_custo_uniforme(inicio, objetivo):
    """Expande sempre o nó de menor custo acumulado g(n).

    Parâmetros:
        inicio (str): cidade de partida.
        objetivo (str): cidade de destino.

    Devolve:
        tuple: (caminho, custo, nós expandidos, ordem de expansão).
    """
    contador = itertools.count()      # gera 0, 1, 2, 3... para desempate
    fronteira = []
    heapq.heappush(fronteira, (0, next(contador), No(inicio)))
    explorados = set()
    ordem = []

    while fronteira:
        # heappop devolve a tupla inteira; "_" marca o que não vamos usar.
        _, _, no = heapq.heappop(fronteira)

        if no.estado == objetivo:
            return reconstruir_caminho(no), no.g, len(ordem), ordem

        # Um mesmo estado pode ter entrado várias vezes com custos diferentes.
        # Se já expandimos este estado, esta cópia é obsoleta: descartamos.
        if no.estado in explorados:
            continue

        explorados.add(no.estado)
        ordem.append(no.estado)

        for vizinho, custo in MAPA[no.estado].items():
            if vizinho in explorados:
                continue

            filho = No(vizinho, pai=no, acao=vizinho, g=no.g + custo)

            # -------------------------------------------------------------
            # TODO 3: empurre o filho no heap.
            # A prioridade desta busca é o custo acumulado do filho.
            # Apague a linha "pass" e escreva sua solução no lugar.
            # -------------------------------------------------------------
            pass

    return None, None, len(ordem), ordem


# ------------------- teste automático, não precisa alterar -------------------
cam_ucs, custo_ucs, exp_ucs, ordem_ucs = busca_custo_uniforme("Arad", "Bucareste")

if cam_ucs is None:
    print("TODO 3 ainda não foi resolvido: a busca não encontrou caminho.")
elif custo_ucs == 418 and exp_ucs == 12:
    print("TODO 3 correto!")
    print("  Caminho:", " -> ".join(cam_ucs))
    print("  Custo:", custo_ucs, "km  |  Nós expandidos:", exp_ucs)
    print()
    print("A busca em largura tinha devolvido 450 km.")
    print("Você acabou de economizar", 450 - custo_ucs, "km.")
else:
    print("Resultado inesperado: custo", custo_ucs, "e", exp_ucs,
          "expandidos - esperado: custo 418 e 12 expandidos.")

### O Diagnóstico do Preço

Compare os dois resultados que você já tem:

| Algoritmo | Custo | Nós expandidos |
|---|---|---|
| Busca em largura | 450 km | 8 |
| Busca de custo uniforme | 418 km | 12 |

A segunda achou um caminho **32 km mais curto**, mas expandiu **50% mais cidades**.

**Isso significa que ela é pior? Não.** Significa que otimalidade tem preço. Para garantir que nenhum caminho mais barato existe, ela precisa expandir *todos* os nós com custo menor que o da solução — inclusive Drobeta, no extremo sudoeste, na direção oposta ao destino.

Como no Laboratório 1, o resultado só faz sentido à luz do critério: lá era a medida de desempenho, aqui é o custo de caminho.

In [ ]:
# ---------------------------------------------------------------------
# Visualizando a busca de custo uniforme: o ótimo, e o preço pago.
# ---------------------------------------------------------------------

if cam_ucs:
    desenhar_mapa(
        caminho=cam_ucs,
        expandidos=ordem_ucs,
        titulo=f"Custo uniforme - {custo_ucs} km, {exp_ucs} nós expandidos",
    )
    print("Ordem de expansão:", " · ".join(ordem_ucs))
    print()
    print("Repare: expandiu Zerind, Timisoara, Lugoj, Mehadia e Drobeta -")
    print("todas na direção OPOSTA a Bucareste. Ela não tem como saber.")
else:
    print("Complete o TODO 3 para ver este gráfico.")

---
# Etapa 6 — Busca em profundidade: o contraexemplo

Esta você **não** precisa escrever: ela já vem pronta, porque o objetivo aqui é outro. É um contraexemplo, e existe para desmontar uma intuição errada.

A busca em profundidade troca a fila por uma **pilha**: sempre expande o nó mais recente. Ela mergulha por um ramo até o fundo antes de considerar alternativas.

Leia o código e repare que a única diferença real para a busca em largura é `.pop()` no lugar de `.popleft()`. Depois olhe o resultado com atenção.

In [ ]:
# ---------------------------------------------------------------------
# Busca em profundidade — já implementada, para servir de contraexemplo.
#
# Diferença para a busca em largura: .pop() em vez de .popleft(). Só isso.
# Empilhamos os vizinhos na ordem do mapa, então o ÚLTIMO vizinho listado
# é o primeiro a ser explorado.
# ---------------------------------------------------------------------

def busca_profundidade(inicio, objetivo):
    """Busca em profundidade em grafo. Não é ótima — e é esse o ponto."""
    pilha = [No(inicio)]
    explorados = set()
    ordem = []

    while pilha:
        no = pilha.pop()                     # retira o mais recente: pilha

        if no.estado == objetivo:
            return reconstruir_caminho(no), no.g, len(ordem), ordem

        if no.estado in explorados:
            continue

        explorados.add(no.estado)
        ordem.append(no.estado)

        for vizinho, custo in MAPA[no.estado].items():
            if vizinho not in explorados:
                pilha.append(No(vizinho, pai=no, acao=vizinho, g=no.g + custo))

    return None, None, len(ordem), ordem


cam_dfs, custo_dfs, exp_dfs, ordem_dfs = busca_profundidade("Arad", "Bucareste")

print("Caminho:", " -> ".join(cam_dfs))
print("Custo:", custo_dfs, "km  |  Nós expandidos:", exp_dfs)
print()
print("O caminho ótimo custa 418 km. Esta busca devolveu", custo_dfs, "km,")
print(f"ou seja, {(custo_dfs / 418 - 1) * 100:.0f}% a mais.")
print("E ela expandiu", exp_dfs, "nós - MENOS que a busca em largura, que expandiu 8.")

### A lição do contraexemplo

A busca em profundidade expandiu **menos nós que todas as outras buscas cegas** e mesmo assim devolveu o **pior caminho do laboratório**.

Isso derruba de vez a ideia de que "expandiu pouco" significa "foi bom". São dois eixos independentes:

- **custo do caminho** mede a *qualidade da resposta*;
- **nós expandidos** mede o *esforço para obtê-la*.

Um algoritmo pode ser barato e ruim, caro e bom, ou — como você vai ver no Desafio Final — barato **e** bom. Só que para isso ele precisa de algo que nenhum algoritmo desta seção tem: **informação sobre onde fica o objetivo**.

---
# Etapa 7 — A heurística e a busca gulosa

## O que é uma heurística

Uma **função heurística** `h(n)` estima o custo do caminho mais barato do estado `n` até o objetivo. Ela é um palpite — mas um palpite com regras.

Para a Romênia usamos a **distância em linha reta** até Bucareste. Note que essa informação **não estava no mapa**: o mapa só tem estradas, e alguém precisou medir as retas a partir das coordenadas. É informação extra, vinda de fora do grafo.

## A propriedade que faz tudo funcionar

Uma heurística é **admissível** quando nunca superestima:

$$h(n) \le h^*(n)$$

onde $h^*(n)$ é o custo **verdadeiro** do melhor caminho de `n` até o objetivo. A distância em linha reta é admissível por um motivo geométrico simples: **nenhuma estrada é mais curta que a linha reta entre dois pontos**. Voar sempre custa menos ou igual a dirigir.

Mas não vamos aceitar isso na base da confiança. Vamos **verificar no código**, cidade por cidade — e a ferramenta para calcular $h^*$ é justamente a busca de custo uniforme que você escreveu no TODO 3.

In [ ]:
# ---------------------------------------------------------------------
# A heurística: distância em linha reta de cada cidade até Bucareste.
# Fonte: Russell & Norvig, 3ª ed., Figura 3.22.
# ---------------------------------------------------------------------

HSLD = {
    "Arad": 366, "Bucareste": 0, "Craiova": 160, "Drobeta": 242,
    "Eforie": 161, "Fagaras": 176, "Giurgiu": 77, "Hirsova": 151,
    "Iasi": 226, "Lugoj": 244, "Mehadia": 241, "Neamt": 234,
    "Oradea": 380, "Pitesti": 100, "Rimnicu Vilcea": 193, "Sibiu": 253,
    "Timisoara": 329, "Urziceni": 80, "Vaslui": 199, "Zerind": 374,
}

# Verificação de sanidade: toda cidade do mapa tem valor de heurística?
faltando = [c for c in MAPA if c not in HSLD]
print("Cidades sem heurística:", faltando if faltando else "nenhuma")

# O objetivo precisa ter h = 0: quando chegamos, não falta mais nada.
print("h(Bucareste) =", HSLD["Bucareste"], "- tem que ser zero")
print()

print("Algumas leituras:")
for cidade in ["Arad", "Sibiu", "Fagaras", "Rimnicu Vilcea", "Pitesti"]:
    print(f"  h({cidade:16s}) = {HSLD[cidade]:3d} km em linha reta")

In [ ]:
# ---------------------------------------------------------------------
# PROVA EMPÍRICA DA ADMISSIBILIDADE.
#
# Para cada cidade comparamos:
#   h(cidade)  -> a estimativa em linha reta
#   h*(cidade) -> o menor custo real por estrada, calculado com o TODO 3
#
# Se em ALGUMA cidade a estimativa for maior que o custo real, a
# heurística não é admissível e o A* perde a garantia de otimalidade.
# ---------------------------------------------------------------------

violacoes = []
linhas_tabela = []

for cidade in sorted(MAPA):
    if cidade == "Bucareste":
        continue

    # Reusamos o TODO 3: ele devolve o custo real mínimo até Bucareste.
    _, custo_real, _, _ = busca_custo_uniforme(cidade, "Bucareste")

    if custo_real is None:              # o TODO 3 ainda não foi completado
        print("Complete o TODO 3 antes de rodar esta célula.")
        break

    estimativa = HSLD[cidade]
    folga = custo_real - estimativa     # quanto a heurística subestimou
    linhas_tabela.append((cidade, estimativa, custo_real, folga))

    if estimativa > custo_real:
        violacoes.append(cidade)

if linhas_tabela:
    print(f"{'Cidade':18s} {'h(n)':>6s} {'h*(n)':>6s} {'folga':>7s}")
    print("-" * 40)
    # sorted com key=... ordena pela folga, do menor para o maior.
    for cidade, est, real, folga in sorted(linhas_tabela, key=lambda t: t[3]):
        print(f"{cidade:18s} {est:6d} {real:6d} {folga:7d}")

    print()
    if violacoes:
        print("Heurística NÃO admissível. Violações em:", violacoes)
    else:
        print(f"Admissível nas {len(linhas_tabela)} cidades: h nunca superou h*.")
        apertada = min(linhas_tabela, key=lambda t: t[3])
        print(f"Caso mais apertado: {apertada[0]}, com folga de apenas "
              f"{apertada[3]} km.")

### O que a coluna "folga" está dizendo

A folga é a diferença entre o custo real e a estimativa — ou seja, o quanto a heurística foi conservadora.

- Em **Pitesti** a folga é de apenas 1 km: a estimativa é 100 e o custo real é 101. É quase perfeita.
- Em cidades distantes, como Timisoara, a folga passa de 200 km: a linha reta ignora que é preciso contornar meio país.

**Quanto menor a folga, mais informada é a heurística** — e menos nós o algoritmo vai precisar expandir. Uma heurística com folga zero em todas as cidades levaria direto ao objetivo. Uma heurística com `h(n) = 0` em todas também é admissível (nunca superestima!), mas não informa nada — e nesse caso o A\* degenera exatamente na busca de custo uniforme.

> **Guarde isto para o Marco 1:** quando o seu trio inventar a heurística do domínio de vocês, a pergunta a fazer é sempre a mesma — *ela pode, em algum caso, chutar mais alto que o custo real?* Se puder, ela não é admissível.

### A busca gulosa: a ideia mais óbvia, e errada

A **busca gulosa** ordena a fronteira apenas por `h`. A cada passo, vai para o vizinho que **parece** mais perto do objetivo. Ela vem pronta abaixo, porque o que interessa aqui é observar **onde** ela erra.

In [ ]:
# ---------------------------------------------------------------------
# Busca gulosa — já implementada. Ordena a fronteira SÓ por h(n).
#
# Repare na linha do heappush: a prioridade é h[vizinho], e o custo já
# percorrido (filho.g) não entra na conta. Esse esquecimento é o defeito.
# ---------------------------------------------------------------------

def busca_gulosa(inicio, objetivo, h):
    """Ordena a fronteira apenas pela estimativa h(n)."""
    contador = itertools.count()
    fronteira = []
    heapq.heappush(fronteira, (h[inicio], next(contador), No(inicio)))
    explorados = set()
    ordem = []

    while fronteira:
        _, _, no = heapq.heappop(fronteira)

        if no.estado == objetivo:
            return reconstruir_caminho(no), no.g, len(ordem), ordem

        if no.estado in explorados:
            continue

        explorados.add(no.estado)
        ordem.append(no.estado)

        for vizinho, custo in MAPA[no.estado].items():
            if vizinho in explorados:
                continue
            filho = No(vizinho, pai=no, acao=vizinho, g=no.g + custo)
            # A prioridade é SÓ a heurística: o passado é ignorado.
            heapq.heappush(fronteira, (h[vizinho], next(contador), filho))

    return None, None, len(ordem), ordem


cam_gul, custo_gul, exp_gul, ordem_gul = busca_gulosa("Arad", "Bucareste", HSLD)

print("Caminho:", " -> ".join(cam_gul))
print("Custo:", custo_gul, "km  |  Nós expandidos:", exp_gul)
print()
print("Apenas", exp_gul, "expansões: o menor esforço de todo o laboratório.")
print("E", custo_gul, "km: o mesmo caminho subótimo da busca em largura.")

### Onde exatamente a gulosa errou

O erro acontece em **um único ponto**: a saída de Sibiu.

Em Sibiu o algoritmo tem duas opções relevantes:

| Destino | `g` (já percorrido) | `h` (linha reta) | `g + h` |
|---|---|---|---|
| Fagaras | 140 + 99 = **239** | 176 | 415 |
| Rimnicu Vilcea | 140 + 80 = **220** | 193 | **413** |

A gulosa compara **só a coluna `h`**: 176 é menor que 193, então vai para Fagaras. Mas chegar a Fagaras já custou 19 km a mais do que chegar a Rimnicu Vilcea — e a gulosa esqueceu esse detalhe, porque ela não olha para trás.

Olhe a última coluna. Somando as duas parcelas, **a ordem se inverte**: 413 é menor que 415.

Corrigir isso é o Desafio Final.

In [ ]:
# ---------------------------------------------------------------------
# Conferindo a armadilha de Sibiu com os números do próprio código.
# ---------------------------------------------------------------------

g_sibiu = MAPA["Arad"]["Sibiu"]            # 140 km, já percorridos

for destino in ["Fagaras", "Rimnicu Vilcea"]:
    g = g_sibiu + MAPA["Sibiu"][destino]   # custo real até o destino
    h = HSLD[destino]                      # estimativa do que falta
    print(f"{destino:16s}  g = {g:3d}   h = {h:3d}   g + h = {g + h:3d}")

print()
print("Olhando só h    -> 176 < 193, vence Fagaras.")
print("Olhando g + h   -> 413 < 415, vence Rimnicu Vilcea.")
print("A ordem se inverte. Essa inversão é o A*.")

---
# Etapa 8 — Desafio Final (Entrega Obrigatória)

## Somando as duas parcelas

Você já viu que olhar só para `g` custa caro (12 expansões) e olhar só para `h` erra o caminho (450 km). O algoritmo **A\*** usa as duas:

$$f(n) = g(n) + h(n)$$

- `g(n)` é o custo **real** já percorrido, do início até `n`;
- `h(n)` é o custo **estimado** que ainda falta, de `n` até o objetivo;
- `f(n)` é a estimativa do custo **total** do melhor caminho que passa por `n`.

## 🛠️ O que você precisa fazer

**Parte 1 — Completar o código (TODO 4):**
Escreva a linha do `heappush` do A\*. É a mesma da busca gulosa, com uma soma a mais.

**Parte 2 — Análise de IA:**
Rode o experimento comparativo e preencha a célula de texto ao final.

**Resultado esperado do TODO 4:** `Arad → Sibiu → Rimnicu Vilcea → Pitesti → Bucareste`, custo **418 km**, **5** nós expandidos.

In [ ]:
# =====================================================================
# TODO 4 — Complete o A*.
#
# Falta UMA linha: empurrar o filho no heap com a prioridade do A*.
# Dica: f(n) = g(n) + h(n)
#       g é o custo já percorrido do filho -> filho.g
#       h é a estimativa do que falta      -> h[vizinho]
# =====================================================================

def a_estrela(inicio, objetivo, h):
    """Ordena a fronteira por f(n) = g(n) + h(n).

    Parâmetros:
        inicio (str): cidade de partida.
        objetivo (str): cidade de destino.
        h (dict): a função heurística, como dicionário cidade -> estimativa.

    Devolve:
        tuple: (caminho, custo, nós expandidos, ordem de expansão).
    """
    contador = itertools.count()
    fronteira = []
    heapq.heappush(fronteira, (h[inicio], next(contador), No(inicio)))
    explorados = set()
    ordem = []

    while fronteira:
        _, _, no = heapq.heappop(fronteira)

        if no.estado == objetivo:
            return reconstruir_caminho(no), no.g, len(ordem), ordem

        if no.estado in explorados:
            continue

        explorados.add(no.estado)
        ordem.append(no.estado)

        for vizinho, custo in MAPA[no.estado].items():
            if vizinho in explorados:
                continue

            filho = No(vizinho, pai=no, acao=vizinho, g=no.g + custo)

            # -------------------------------------------------------------
            # TODO 4: empurre o filho no heap com prioridade f = g + h.
            # Apague a linha "pass" e escreva sua solução no lugar.
            # -------------------------------------------------------------
            pass

    return None, None, len(ordem), ordem


# ------------------- teste automático, não precisa alterar -------------------
cam_ast, custo_ast, exp_ast, ordem_ast = a_estrela("Arad", "Bucareste", HSLD)

if cam_ast is None:
    print("TODO 4 ainda não foi resolvido: a busca não encontrou caminho.")
elif custo_ast == 418 and exp_ast == 5:
    print("TODO 4 correto! Você implementou o A*.")
    print("  Caminho:", " -> ".join(cam_ast))
    print("  Custo:", custo_ast, "km  |  Nós expandidos:", exp_ast)
    print()
    print("Mesmo custo ótimo da busca de custo uniforme (418 km),")
    print("com 5 expansões em vez de 12: menos da metade do trabalho.")
elif custo_ast == 450:
    print("Resultado inesperado: custo 450. Você usou só h - isso é a busca gulosa.")
    print("Some também o custo já percorrido do filho.")
else:
    print("Resultado inesperado: custo", custo_ast, "e", exp_ast,
          "expandidos - esperado: custo 418 e 5 expandidos.")

### Acompanhando f = g + h ao longo do caminho

A célula abaixo abre o valor de `f` em cada cidade do caminho encontrado pelo A\*.

Preste atenção em uma coisa: **`f` nunca diminui**. Ele sobe devagar até encostar no custo real de 418 km. Isso não é coincidência — é a assinatura de uma heurística **consistente**, e é o que garante que, quando o A\* expande um nó, ele já encontrou o melhor caminho até ali.

In [ ]:
# ---------------------------------------------------------------------
# Abrindo f = g + h em cada passo do caminho encontrado pelo A*.
# ---------------------------------------------------------------------

if cam_ast:
    print(f"{'Cidade':18s} {'g':>6s} {'h':>6s} {'f = g+h':>9s}")
    print("-" * 42)

    g_acumulado = 0
    valores_f = []

    # enumerate() devolve (índice, item); usamos o índice para pegar a
    # cidade anterior e somar o custo do trecho.
    for i, cidade in enumerate(cam_ast):
        if i > 0:
            g_acumulado += MAPA[cam_ast[i - 1]][cidade]
        f = g_acumulado + HSLD[cidade]
        valores_f.append(f)
        print(f"{cidade:18s} {g_acumulado:6d} {HSLD[cidade]:6d} {f:9d}")

    print()
    print("Sequência de f:", " -> ".join(str(v) for v in valores_f))

    # all() devolve True se TODAS as comparações forem verdadeiras.
    nao_decresce = all(a <= b for a, b in zip(valores_f, valores_f[1:]))
    print("f nunca diminuiu?", "sim" if nao_decresce else "NÃO")
    print("No objetivo, h = 0, portanto f = g =", valores_f[-1], "km")
else:
    print("Complete o TODO 4 para ver esta análise.")

### O Grande Experimento Comparativo

Como no Laboratório 1, vamos colocar todos os algoritmos sob **condições idênticas**: o mesmo mapa, o mesmo par origem-destino, a mesma convenção de teste de objetivo.

Esse é o rigor técnico exigido nos projetos da disciplina — e é exatamente o formato que o Marco 1 vai cobrar.

In [ ]:
# ---------------------------------------------------------------------
# PAINEL COMPARATIVO — os cinco algoritmos sobre o mesmo problema.
#
# Cada entrada da lista é (nome, função, precisa de heurística?).
# ---------------------------------------------------------------------

import pandas as pd

algoritmos = [
    ("Profundidade", busca_profundidade, False),
    ("Largura", busca_largura, False),
    ("Custo uniforme", busca_custo_uniforme, False),
    ("Gulosa", busca_gulosa, True),
    ("A*", a_estrela, True),
]

linhas_painel = []
for nome, funcao, usa_h in algoritmos:
    # Chamamos com 3 argumentos se a função usa heurística, 2 caso contrário.
    if usa_h:
        caminho, custo, expandidos, _ = funcao("Arad", "Bucareste", HSLD)
    else:
        caminho, custo, expandidos, _ = funcao("Arad", "Bucareste")

    if caminho is None:              # algum TODO ainda em branco
        print(nome, "- TODO ainda não completado.")
        continue

    linhas_painel.append({
        "algoritmo": nome,
        "custo_km": custo,
        "nos_expandidos": expandidos,
        "cidades_no_caminho": len(caminho),
        "otimo": "sim" if custo == 418 else "não",
    })

if linhas_painel:
    painel = pd.DataFrame(linhas_painel)
    print(painel.to_string(index=False))

In [ ]:
# ---------------------------------------------------------------------
# GRÁFICO COMPARATIVO — qualidade e esforço, lado a lado.
#
# Dois subgráficos porque as escalas são diferentes: custo está em
# centenas de quilômetros, esforço está em unidades de nós.
# ---------------------------------------------------------------------

if linhas_painel:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # As cores saem do painel, para o gráfico continuar correto mesmo
    # que algum TODO ainda esteja em branco.
    paleta = {"Profundidade": "#9A6700", "Largura": "#1E4E79",
              "Custo uniforme": "#1B7F5A", "Gulosa": "#5A6570", "A*": "#C8102E"}
    nomes = list(painel["algoritmo"])
    cores = [paleta[a] for a in nomes]

    b1 = ax1.bar(nomes, painel["custo_km"], color=cores)
    ax1.axhline(418, color="#C8102E", ls="--", lw=1.5)
    ax1.set_title("Custo do caminho encontrado", fontsize=13, fontweight="bold")
    ax1.set_ylabel("quilômetros")
    ax1.bar_label(b1, fmt="%d km", fontsize=11, padding=3)
    ax1.set_ylim(0, 850)

    b2 = ax2.bar(nomes, painel["nos_expandidos"], color=cores)
    ax2.set_title("Esforço: nós expandidos", fontsize=13, fontweight="bold")
    ax2.set_ylabel("nós")
    ax2.bar_label(b2, fontsize=11, padding=3)
    ax2.set_ylim(0, 15)

    for eixo in (ax1, ax2):
        eixo.spines["top"].set_visible(False)
        eixo.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.show()

    print("Barra baixa nos DOIS gráficos ao mesmo tempo: só o A*.")
else:
    print("Complete os TODOs para gerar o gráfico.")

### O Diagnóstico do Empate Aparente

Repare que a busca de custo uniforme e o A\* devolveram **exatamente o mesmo caminho e o mesmo custo**: 418 km.

**Isso prova que os dois algoritmos são equivalentes? Não.**

Isso prova que o **custo do caminho**, sozinho, é uma medida limitada — exatamente como a medida de desempenho do Laboratório 1 era limitada por não punir o movimento. Os dois algoritmos empatam em qualidade e se separam em **esforço**: 12 expansões contra 5.

Como vimos na aula teórica: **a avaliação de um algoritmo depende do critério que você escolheu medir.** Um critério só quase nunca conta a história inteira.

### O experimento estendido: o A\* compensa em qualquer rota?

Arad → Bucareste é um caso só. Antes de generalizar, vale testar várias origens.

In [ ]:
# ---------------------------------------------------------------------
# O A* compensa em qualquer rota, ou só na de Arad?
#
# Testamos vários pares origem-destino. Se em alguma rota o A* devolvesse
# um custo diferente do ótimo, apareceria aqui.
# ---------------------------------------------------------------------

pares = [
    ("Arad", "Bucareste"), ("Oradea", "Bucareste"), ("Timisoara", "Bucareste"),
    ("Neamt", "Bucareste"), ("Zerind", "Bucareste"), ("Craiova", "Bucareste"),
]

resultados = []
for origem, destino in pares:
    _, custo_u, exp_u, _ = busca_custo_uniforme(origem, destino)
    _, custo_a, exp_a, _ = a_estrela(origem, destino, HSLD)

    if custo_u is None or custo_a is None:
        print("Complete os TODOs 3 e 4 antes de rodar o experimento.")
        break

    resultados.append({
        "rota": f"{origem} -> {destino}",
        "custo_uniforme": custo_u,
        "custo_a_estrela": custo_a,
        "mesmo_custo": "sim" if custo_u == custo_a else "NÃO",
        "nos_uniforme": exp_u,
        "nos_a_estrela": exp_a,
        "economia_esforco": f"{(1 - exp_a / exp_u) * 100:.0f}%",
    })

if resultados:
    tabela = pd.DataFrame(resultados)
    print(tabela.to_string(index=False))
    print()

    # Verificação central: o A* achou o ótimo em TODAS as rotas?
    todos_otimos = all(r["mesmo_custo"] == "sim" for r in resultados)
    print("O A* devolveu o custo ótimo em todas as rotas?",
          "sim" if todos_otimos else "NÃO")

    media = sum(int(r["economia_esforco"].rstrip("%")) for r in resultados)
    print(f"Economia média de nós expandidos: {media / len(resultados):.0f}%")

### Sua Análise — Edite esta célula

**Cenário:** uma transportadora que opera na Romênia quer saber se vale a pena investir em levantar as distâncias em linha reta de todas as cidades — um trabalho caro de cartografia — ou se basta usar o roteirizador por custo uniforme que ela já tem.

*Escreva de 6 a 10 linhas estruturando seu raciocínio da seguinte forma:*

1. **Os dados numéricos:** o A\* devolveu o mesmo custo que a busca de custo uniforme em todas as rotas testadas? Qual foi a economia média de nós expandidos?
2. **A lógica subjacente:** qual propriedade da heurística garante que o A\* não devolveria um caminho pior? Cite o nome dela e a verificação que você rodou na Etapa 7.
3. **O contraexemplo interno:** por que a busca gulosa, que usa a *mesma* heurística e expandiu ainda menos nós, não serve como resposta para a transportadora?
4. **A ressalva honesta:** o mapa da Romênia tem 20 cidades. Cite uma limitação do A\* que não aparece neste laboratório e que apareceria com 200 mil cidades.
5. **A recomendação:** o que você diria à transportadora, e sob qual condição a resposta mudaria.

---

_Escreva sua análise aqui, substituindo esta linha._

---

---
# Encerramento

## ✅ Checklist de Entrega

Antes de postar no Google Sala de Aula, certifique-se de que:

- [ ] O **TODO 1** foi completado (`custo_do_caminho` devolvendo 450 e 418).
- [ ] O **TODO 2** foi completado (busca em largura: 450 km, 8 nós).
- [ ] O **TODO 3** foi completado (custo uniforme: 418 km, 12 nós).
- [ ] O **TODO 4** foi completado (A\*: 418 km, 5 nós).
- [ ] A célula Markdown de **Análise** final foi preenchida.
- [ ] O menu **`Ambiente de execução → Reiniciar sessão e executar tudo`** encerrou sem erros! (Evite perder ponto em reprodutibilidade.)

## Resumo de Fixação (De/Para)

| Teoria de Inteligência Artificial | Implementação no Código Python |
|---|---|
| Estado | Uma string: `"Pitesti"` |
| Nó | Objeto da classe `No` |
| Ações e modelo de transição | `MAPA[cidade].items()` |
| Custo de passo | `MAPA[cidade][vizinho]` |
| Custo de caminho g(n) | O atributo `no.g` |
| Fronteira (fila) | `deque` com `.popleft()` |
| Fronteira (prioridade) | `heapq` com tuplas `(f, contador, nó)` |
| Conjunto de explorados | Um `set` de strings |
| Heurística h(n) | O dicionário `HSLD` |
| Função de avaliação f(n) | `filho.g + h[vizinho]` no `heappush` |

## O que você construiu hoje

Em pouco mais de cem linhas, você escreveu quatro algoritmos de busca que compartilham o mesmo esqueleto e diferem em **uma única linha** cada um — a que insere na fronteira. Essa é a lição estrutural do laboratório: **a estrutura de dados escolhida é o algoritmo**.

E você verificou empiricamente uma propriedade matemática: a admissibilidade da heurística, cidade por cidade, usando o seu próprio código de custo uniforme como referência.

---
**Para a Próxima Aula:**
- A Aula 4 é a **oficina do Projeto Acadêmico**. Traga o **domínio escolhido pelo seu trio** e a formulação em cinco componentes já escrita.
- O **Marco 1** pede exatamente o que você acabou de fazer, aplicado ao domínio de vocês: uma busca não informada, o A\* com heurística **própria**, a comparação numérica entre as duas e o **argumento de por que a heurística é admissível**. Esse último item é o que mais pesa e o que mais gente esquece.

---